### Constant-NVE Simulation in an Extended Phase-Space

This example aims to demonstrate the correctness of the implemented Extended Phase-Space
(XPS) simulation approach by checking energy conservation.

In [ ]:
import cvpack
import numpy as np
import openmm as mm
import openxps as xps
import pandas as pd

from openmm import app, unit
from openmmtools import testsystems
from matplotlib import pyplot as plt

The physical system is an alanine dipeptide molecule in a vacuum and the simulation is
performed in the NVE ensemble using the Verlet integrator.

In [ ]:
model = testsystems.AlanineDipeptideVacuum()
physical_integrator = mm.VerletIntegrator(1 * unit.femtoseconds)
platform = mm.Platform.getPlatformByName("Reference")
simulation = app.Simulation(model.topology, model.system, physical_integrator, platform)
simulation.context.setPositions(model.positions)
simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)

The two Ramachandran angles $\phi({\bf r})$ and $\psi({\bf r})$ are taken as collective
variables (CVs) and associated with two new degrees of freedom (DOFs) $\phi_s$ and
$\psi_s$, respectively.

In [ ]:
backbone_atoms = model.mdtraj_topology.select("name C CA N")
phi = cvpack.Torsion(*backbone_atoms[0:4], name="phi")
psi = cvpack.Torsion(*backbone_atoms[1:5], name="psi")

mass = 3 * unit.dalton * (unit.nanometer / unit.radians)**2
phi_s = xps.ExtraDOF("phi_s", unit.radian, mass, xps.bounds.CIRCULAR)
psi_s = xps.ExtraDOF("psi_s", unit.radian, mass, xps.bounds.CIRCULAR)

The coupling between the CVs and extra DOFs is achieved by adding a harmonic potential
to the Hamiltonian.

In [ ]:
coupling_potential = cvpack.MetaCollectiveVariable(
    "0.5*kappa*(phi_dist^2+psi_dist^2)"
    f"; phi_dist=min(delta_phi,{2*np.pi}-delta_phi)"
    f"; psi_dist=min(delta_psi,{2*np.pi}-delta_psi)"
    "; delta_phi=abs(phi-phi_s)"
    "; delta_psi=abs(psi-psi_s)",
    [phi, psi],
    unit.kilojoule_per_mole,
    kappa=1000 * unit.kilojoule_per_mole / unit.radian**2,
    phi_s=np.pi * unit.radian,
    psi_s=np.pi * unit.radian,
    name="coupling_potential",
)

To execute the XPS simulation, the original context is transformed into an extended
space context by adding the extra DOFs and the coupling potential.

In [ ]:
context = xps.ExtendedSpaceContext(
    simulation.context, [phi_s, psi_s], coupling_potential,
)
cv_values = coupling_potential.getInnerValues(context)
context.setExtraValues([cv_values["phi"], cv_values["psi"]])
context.setExtraVelocitiesToTemperature(300 * unit.kelvin)

The total energy consists of the potential energy of the physical system, which includes
the harmonic coupling potential, and the kinetic energy of both the physical and extra
DOFs.

In [ ]:
interval = 100
cv_reporter = cvpack.reporting.StateDataReporter(
    "data.csv",
    interval,
    step=True,
    potentialEnergy=True,
    kineticEnergy=True,
    writers=[
        xps.StateDataWriter(
            context.getExtensionState,
            prefix="Extra",
            kineticEnergy=True,
        ),
        cvpack.reporting.MetaCVWriter(
            coupling_potential,
            values=["phi", "psi"],
            parameters=["phi_s", "psi_s"],
        ),
    ],
    speed=True,
)
simulation.reporters = [cv_reporter]

Running the simulation for a long time, the total energy should be conserved, which
demonstrates the correctness of the XPS simulation approach.

In [ ]:
simulation.step(100000)
data = pd.read_csv("data.csv")

In [ ]:

steps = data['#"Step"']
potential = data["Potential Energy (kJ/mole)"]
physical_kinetic = data["Kinetic Energy (kJ/mole)"]
extra_kinetic = data["Extra Kinetic Energy (kJ/mole)"]
total = potential + physical_kinetic + extra_kinetic

fig, ax = plt.subplots()
ax.plot(steps, potential, label="Potential Energy")
ax.plot(steps, physical_kinetic, label="Physical Kinetic Energy")
ax.plot(steps, extra_kinetic, label="Extra Kinetic Energy")
ax.plot(steps, total, label="Total Energy")
# place the legend outside the plot
fig.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xlabel('Step')
ax.set_ylabel('Energy (kJ/mol)')
ax.set_title("Constant NVE Simulation in an Extended Phase Space")
plt.show()

In [ ]:
fig, ax = plt.subplots()

def unwrapped(angles):
    angles = np.unwrap(angles)
    if angles.mean() < - np.pi:
        return angles + 2 * np.pi
    if angles.mean() > np.pi:
        return angles - 2 * np.pi
    return angles

ax.plot(steps, unwrapped(data["phi (rad)"]), label=r"$\phi({\bf r})$")
ax.plot(steps, unwrapped(data["psi (rad)"]), label=r"$\psi({\bf r})$")
ax.plot(steps, unwrapped(data["phi_s (rad)"]), label=r"$\phi_s$")
ax.plot(steps, unwrapped(data["psi_s (rad)"]), label=r"$\psi_s$")
# place the legend outside the plot
fig.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xlabel('Step')
ax.set_ylabel('Angle (radians)')
ax.set_title("Constant NVE Simulation in an Extended Phase Space")
plt.show()